# 02 - Fine-tune XLM-RoBERTa-base for Pakistani NER
Runtime > Change runtime type > **T4 GPU** first. Then run the cells top to bottom.
Upload `outputs/colab_bundle.zip` (made on your PC) when asked in cell 3.

In [ ]:
!nvidia-smi | head -n 12

In [ ]:
!pip -q install -U transformers accelerate

In [ ]:
from google.colab import files
up = files.upload()            # choose colab_bundle.zip
!unzip -oq colab_bundle.zip
%cd ner
!ls data/dataset

### Smoke test (about 2 minutes) - checks that data, labels and metrics work before the real run

In [ ]:
!python scripts/train_ner.py --smoke --out outputs/smoke

### Real run (about 25-45 min on a T4)

In [ ]:
!python scripts/train_ner.py --model xlm-roberta-base --out outputs/xlmr_v3 --epochs 6

### Save results to Google Drive (Colab deletes files when the session ends)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ner_results
!cp -r outputs/xlmr_v3 /content/drive/MyDrive/ner_results/
!ls /content/drive/MyDrive/ner_results/xlmr_v3

Download `report_test.txt`, `report_extra_test.txt`, `errors_test.txt`, `errors_extra_test.txt` and put them in `outputs/` on your PC for error analysis.

## Optional: check the installed versions (copy the output into PROJECT_LOG.md)

In [ ]:
!pip list 2>/dev/null | grep -i -E "^(torch|transformers|accelerate|numpy) "

## Test the inference script (`predict.py`) on the trained model
If the session is still alive the model is in `outputs/xlmr_v3/best`. If you restarted, run the next cell first to copy it back from Drive.

In [ ]:
import os
if not os.path.exists('outputs/xlmr_v3/best'):
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p outputs/xlmr_v3
    !cp -r /content/drive/MyDrive/ner_results/xlmr_v3/best outputs/xlmr_v3/
!ls outputs/xlmr_v3/best

In [ ]:
!python scripts/predict.py --model outputs/xlmr_v3/best --text "Prime Minister Shehbaz Sharif met Imran Khan of PTI in Lahore on 5 March 2024; call 0300-1234567, CNIC 35202-1234567-1."

In [ ]:
open('my_sentences.txt', 'w').write(
    'The Supreme Court of Pakistan reserved its verdict on Tuesday.\n'
    'Dawn reported that the Sindh Assembly passed the Elections Act 2017 amendment.\n'
    'NED University of Engineering and Technology announced a Rs 5 million scholarship.\n')
!python scripts/predict.py --model outputs/xlmr_v3/best --input my_sentences.txt --out predictions.jsonl
!head -c 600 predictions.jsonl

## Score ANY saved model on all test sets (no training): e.g. to check v2 on the fixed pattern test
Set the folder of the saved model (`.../best`) and an output folder name.

In [ ]:
!python scripts/train_ner.py --eval_only outputs/xlmr_v3/best --out outputs/eval_v3